<a href="https://colab.research.google.com/github/inosven/Neural-Networks-Zero-to-Hero/blob/main/makemore_Part_2_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
with open('/content/drive/MyDrive/Colab Notebooks/Zero2Hero_Karpathy/Data/names.txt','r') as f:
  text = f.read().splitlines()

Mounted at /content/drive


In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
words = text
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [10]:
# build the volcabulary of charaters and mappings to/from integers
chars = sorted(list(set(''.join(words))))

stoi = {s:i+1 for i , s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [11]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '------------>', itos[ix])
    context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ------------> e
..e ------------> m
.em ------------> m
emm ------------> a
mma ------------> .
olivia
... ------------> o
..o ------------> l
.ol ------------> i
oli ------------> v
liv ------------> i
ivi ------------> a
via ------------> .
ava
... ------------> a
..a ------------> v
.av ------------> a
ava ------------> .
isabella
... ------------> i
..i ------------> s
.is ------------> a
isa ------------> b
sab ------------> e
abe ------------> l
bel ------------> l
ell ------------> a
lla ------------> .
sophia
... ------------> s
..s ------------> o
.so ------------> p
sop ------------> h
oph ------------> i
phi ------------> a
hia ------------> .


In [12]:
X.shape, X.dtype, Y.shape, Y.dtype,

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [26]:
X[1]

tensor([0, 0, 5])

In [13]:
C = torch.randn((27,2))

In [34]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [35]:
F.one_hot(torch.tensor(5), num_classes=27).float()

tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [32]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [38]:
torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1).shape

torch.Size([32, 6])

In [42]:
len(torch.unbind(emb,1))

3

In [45]:
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [48]:
a = torch.arange(18)
a.shape

torch.Size([18])

In [50]:
a.view(3,3, 2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [51]:
a.storage()

/tmp/ipykernel_4032/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [52]:
emb.shape

torch.Size([32, 3, 2])

In [54]:
emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [61]:
h = torch.tanh(emb.view(-1,emb.shape[1]*emb.shape[2]) @ W1 + b1)
h.shape

torch.Size([32, 100])

In [60]:
h.shape

torch.Size([32, 100])

In [62]:
W2= torch.randn((100,27))
b2 = torch.rand(27)

In [63]:
logits = h @ W2 +b2

In [65]:
logits.shape

tensor([[-3.0659e+00,  1.0768e+01,  1.5295e+01,  6.0333e+00, -9.9170e+00,
         -7.9481e+00,  6.9379e+00,  6.9393e-01, -5.7465e+00, -1.2047e+00,
          1.9431e+00,  6.0117e+00, -1.6771e+00,  9.5746e+00,  4.0384e+00,
         -1.4409e+01,  1.7757e+01, -1.0619e+00,  2.0010e+00,  5.7844e-01,
         -5.8731e+00,  7.6345e+00, -1.6147e+01, -7.4250e+00,  1.0924e+01,
          1.2689e+01, -1.6914e+01],
        [ 1.3945e+00,  1.7957e+01,  1.7928e+01, -6.1443e+00,  1.4234e+00,
          1.2077e+01,  5.2350e-01, -4.9591e+00,  3.4967e+00,  1.1895e+01,
         -3.4876e-01,  1.7263e+01,  1.0489e+01, -1.9927e-02,  1.3535e+01,
         -8.3407e+00, -1.4862e+00, -1.2171e+01, -8.7304e+00, -7.8794e+00,
         -8.8038e+00,  7.1369e+00, -4.7522e+00, -2.9982e+00,  9.4691e+00,
          5.7412e+00, -8.9800e+00],
        [-4.8127e+00,  1.3218e+01, -3.2853e+00,  5.0125e+00, -7.8661e+00,
         -8.4732e+00,  8.3123e+00,  1.1579e+00,  5.7169e+00, -2.1279e-01,
          5.2240e-01, -6.5884e+00, -7.08

In [66]:
logits.shape

torch.Size([32, 27])

In [79]:
counts = logits.exp()
probs = counts / counts.sum(1,keepdim=True)
probs[0]

tensor([8.2736e-10, 8.4264e-04, 7.7963e-02, 7.4035e-06, 8.7561e-13, 6.2714e-12,
        1.8293e-05, 3.5528e-08, 5.6691e-11, 5.3213e-09, 1.2390e-07, 7.2450e-06,
        3.3176e-09, 2.5549e-04, 1.0070e-06, 9.8011e-15, 9.1413e-01, 6.1379e-09,
        1.3129e-07, 3.1653e-08, 4.9951e-11, 3.6712e-05, 1.7242e-15, 1.0582e-11,
        9.8502e-04, 5.7538e-03, 8.0106e-16])

In [74]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [75]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [82]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(16.1198)

In [91]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.rand(27,generator=g)
parameters = [C, W1, b1, W2, b2]

In [92]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [93]:
emb = C[X]
h = torch.tanh(emb.view(-1,emb.shape[1]*emb.shape[2]) @ W1 +b1)
logits = h @ W2 +b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss = - probs[torch.arange(32), Y].log().mean()
loss

tensor(17.0827)

In [94]:
F.cross_entropy(logits, Y)

tensor(17.0827)

In [98]:
for p in parameters:
  p.requires_grad=True
for _ in range(100):
  # forward pass
  emb = C[X] # (32,3,2)
  h = torch.tanh(emb.view(-1,6) @ W1 +b1) # 32* 100
  logits = h @ W2 + b2 # 32*27
  loss= F.cross_entropy(logits, Y)
  # backward pass
  for p in parameters:
    p.grad=None
  loss.backward()
  # udpate
  for p in parameters:
    p.data += -0.1*p.grad
  print(loss.item())

3.4876365661621094
3.153843641281128
2.8603906631469727
2.604771852493286
2.3825836181640625
2.189143657684326
2.0213112831115723
1.8774279356002808
1.7560012340545654
1.6541633605957031
1.56731379032135
1.4907050132751465
1.4209233522415161
1.3558917045593262
1.2943365573883057
1.2354443073272705
1.1787238121032715
1.1239651441574097
1.071225643157959
1.0207667350769043
0.9729203581809998
0.9279274940490723
0.8858373165130615
0.846509575843811
0.8096892833709717
0.7750856280326843
0.7424247860908508
0.7114768624305725
0.6820604205131531
0.654049813747406
0.6273733377456665
0.6020225882530212
0.578062117099762
0.5556351542472839
0.5349501371383667
0.5162321329116821
0.4996369183063507
0.4851664900779724
0.47265127301216125
0.46180614829063416
0.45231887698173523
0.4439104497432709
0.4363575875759125
0.4294903576374054
0.4231828451156616
0.4173409640789032
0.41189318895339966
0.40678584575653076
0.40197688341140747
0.39743295311927795
0.39312776923179626
0.3890393376350403
0.38515019416